In [6]:
!pip -q install requests pandas psycopg2-binary

import requests
import pandas as pd
import psycopg2
from psycopg2.extras import RealDictCursor

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 39.9 MB/s eta 0:00:00


In [7]:
BASE = "https://anapioficeandfire.com/api"

def fetch_all(endpoint: str, params: dict | None = None, page_size: int = 50) -> list[dict]:
    """Постраничная загрузка всех объектов."""
    data, page = [], 1
    params = dict(params or {})
    while True:
        q = params.copy()
        q.update({"page": page, "pageSize": page_size})
        r = requests.get(f"{BASE}/{endpoint}", params=q, timeout=20)
        r.raise_for_status()
        part = r.json()
        if not part:
            break
        data.extend(part)
        page += 1
    return data

books = fetch_all("books")
df_books = pd.DataFrame(books)
print(f"Книги: {df_books.shape}")
display(df_books.head(3))

houses = fetch_all("houses")
df_houses = pd.DataFrame(houses)
print(f"Дома: {df_houses.shape}")
display(df_houses.head(3))

houses_with_words = fetch_all("houses", params={"hasWords": "true"})
df_houses_with_words = pd.DataFrame(houses_with_words)
print(f"Дома с девизом: {df_houses_with_words.shape}")
display(df_houses_with_words[["name", "region", "words"]].head(10))

Книги: (12, 11)


,url,name,isbn,authors,numberOfPages,publisher,country,mediaType,released,characters,povCharacters
0,https://anapioficeandfire.com/api/books/1,A Game of Thrones,978-0553103540,[George R. R. Martin],694,Bantam Books,United States,Hardcover,1996-08-01T00:00:00,[https://anapioficeandfire.com/api/characters/...,[https://anapioficeandfire.com/api/characters/...
1,https://anapioficeandfire.com/api/books/2,A Clash of Kings,978-0553108033,[George R. R. Martin],768,Bantam Books,United States,Hardback,1999-02-02T00:00:00,[https://anapioficeandfire.com/api/characters/...,[https://anapioficeandfire.com/api/characters/...
2,https://anapioficeandfire.com/api/books/3,A Storm of Swords,978-0553106633,[George R. R. Martin],992,Bantam Books,United States,Hardcover,2000-10-31T00:00:00,[https://anapioficeandfire.com/api/characters/...,[https://anapioficeandfire.com/api/characters/...


Дома: (444, 16)


,url,name,region,coatOfArms,words,titles,seats,currentLord,heir,overlord,founded,founder,diedOut,ancestralWeapons,cadetBranches,swornMembers
0,https://anapioficeandfire.com/api/houses/1,House Algood,The Westerlands,"A golden wreath, on a blue field with a gold b...",,[],[],,,https://anapioficeandfire.com/api/houses/229,,,,[],[],[]
1,https://anapioficeandfire.com/api/houses/2,House Allyrion of Godsgrace,Dorne,"Gyronny Gules and Sable, a hand couped Or",No Foe May Pass,[],[Godsgrace],https://anapioficeandfire.com/api/characters/298,https://anapioficeandfire.com/api/characters/1922,https://anapioficeandfire.com/api/houses/285,,,,[],[],[https://anapioficeandfire.com/api/characters/...
2,https://anapioficeandfire.com/api/houses/3,House Amber,The North,,,[],[],,,,,,,[],[],[]


Дома с девизом: (68, 16)


,name,region,words
0,House Allyrion of Godsgrace,Dorne,No Foe May Pass
1,House Ambrose,The Reach,Never Resting
2,House Arryn of the Eyrie,The Vale,As High as Honor
3,House Ashford of Ashford,The Reach,Our Sun Shines Bright
4,House Baratheon of Storm's End,The Stormlands,Ours is the Fury
5,House Beesbury of Honeyholt,The Reach,Beware our Sting
6,House Bolton of the Dreadfort,The North,Our Blades are Sharp
7,House Buckwell of the Antlers,The Crownlands,Pride and Purpose
8,House Bulwer of Blackcrown,The Reach,Death Before Disgrace
9,House Caron of Nightsong,The Stormlands,No Song so Sweet


In [8]:
# Данные из https://rnacentral.org/help/public-database
PG_HOST = "hh-pgsql-public.ebi.ac.uk"
PG_PORT = 5432
PG_DB   = "pfmegrnargs"
PG_USER = "reader"
PG_PASS = "NWDMCE5xdipIjRrp"

def connect_pg(sslmode="prefer"):
    return psycopg2.connect(
        host=PG_HOST,
        port=PG_PORT,
        dbname=PG_DB,
        user=PG_USER,
        password=PG_PASS,
        sslmode=sslmode
    )

def query_df(conn, sql: str) -> pd.DataFrame:
    with conn.cursor(cursor_factory=RealDictCursor) as cur:
        cur.execute(sql)
        rows = cur.fetchall()
    return pd.DataFrame(rows)

try:
    conn = connect_pg("prefer")
    print("Подключились с sslmode='prefer'")
except Exception as e1:
    print("Не удалось с 'prefer':", e1)
    try:
        conn = connect_pg("disable")
        print("Подключились с sslmode='disable'")
    except Exception as e2:
        print("Подключение не удалось:", e2)
        conn = None

if conn:
    # Первые 10 строк из rnc_database
    sql_all = "SELECT * FROM rnc_database LIMIT 10;"
    df_all = query_df(conn, sql_all)
    print("\n=== Первые 10 строк rnc_database ===")
    display(df_all)

    # Определённые столбцы
    sql_sel = """
        SELECT display_name, num_sequences, num_organisms, url
        FROM rnc_database
        LIMIT 10;
    """
    df_sel = query_df(conn, sql_sel)
    print("\n=== Выбранные столбцы ===")
    display(df_sel)

    conn.close()
    print("Соединение закрыто.")
else:
    print("Пропускаем SQL-запросы — нет соединения.")

Подключились с sslmode='prefer'

=== Первые 10 строк rnc_database ===


,id,timestamp,userstamp,descr,current_release,full_descr,alive,for_release,display_name,project_id,avg_length,min_length,max_length,num_sequences,num_organisms,description,url,example,reference
0,21,2017-05-02,RNACEN,NONCODE,146,NONCODE,Y,,NONCODE,,1130.0,201.0,244296.0,234669,7,is an integrated knowledge database dedicated ...,http://www.noncode.org/,"[{'upi': 'URS000019B796', 'taxid': 9606}, {'up...",[{'title': 'NONCODE 2016: an informative and v...
1,5,2017-05-17,RNACEN,VEGA,98,VEGA,N,,VEGA,PRJEB4568,NaN,NaN,NaN,0,0,is a repository for high-quality gene models p...,http://vega.sanger.ac.uk/,"[{'upi': 'URS00000B15DA', 'taxid': 9606}, {'up...",[{'title': 'The GENCODE v7 catalog of human lo...
2,26,2017-05-01,RNACEN,GENCODE,450,GENCODE,N,,GENCODE,,889.0,32.0,205012.0,47677,2,produces high quality reference gene annotatio...,http://gencodegenes.org/,"[{'upi': 'URS00000B15DA', 'taxid': 9606}, {'up...",[{'title': 'GENCODE: the reference human genom...
3,1,2017-05-01,RNACEN,ENA,968,ENA,Y,,ENA,,412.0,10.0,900074.0,12086180,814855,provides a comprehensive record of the world's...,https://www.ebi.ac.uk/ena/browser/,"[{'upi': 'URS00002D0E0C', 'taxid': 10090}, {'u...",[{'title': 'The European Nucleotide Archive in...
4,14,2017-05-01,RNACEN,TAIR,982,TAIR,Y,,TAIR,PRJ_TAIR,384.0,19.0,6227.0,4406,1,is a database of genetic and molecular biology...,http://www.arabidopsis.org/,"[{'upi': 'URS0000591E4F', 'taxid': 3702}, {'up...",[{'title': 'The Arabidopsis Information Resour...
5,9,2017-05-01,RNACEN,REFSEQ,969,RefSeq,Y,,RefSeq,,703.0,15.0,91667.0,120355,22524,"is a comprehensive, integrated, non-redundant,...",http://www.ncbi.nlm.nih.gov/refseq/,"[{'upi': 'URS000075A3E5', 'taxid': 10090}, {'u...",[{'title': 'RefSeq: an update on mammalian ref...
6,41,2017-05-01,RNACEN,GENECARDS,978,MalaCards,Y,,GeneCards,,1292.0,16.0,347561.0,425357,1,"is a searchable, integrative database that pro...",https://www.genecards.org/,"[{'upi': 'URS0000EBFCE3', 'taxid': 9606}, {'up...",[{'title': 'The GeneCards Suite: From Gene Dat...
7,10,2017-05-01,RNACEN,RDP,85,RDP,Y,,RDP,,1536.0,1337.0,1600.0,4779,2487,"provides quality-controlled, aligned and annot...",http://rdp.cme.msu.edu/,"[{'upi': 'URS0000434740', 'taxid': 338963}, {'...",[{'title': 'Ribosomal Database Project: data a...
8,20,2017-05-01,RNACEN,LNCIPEDIA,935,LNCipedia,Y,,LNCipedia,,1534.0,200.0,152544.0,126876,1,is a comprehensive compendium of human long no...,http://www.lncipedia.org/,"[{'upi': 'URS000081175C', 'taxid': 9606}, {'up...",[{'title': 'An update on LNCipedia: a database...
9,15,2017-05-02,RNACEN,WORMBASE,970,WormBase,Y,,WormBase,PRJNA13758,171.0,17.0,84141.0,25550,1,"curates, stores and displays genomic and genet...",http://www.wormbase.org/,"[{'upi': 'URS000022A09E', 'taxid': 6239}, {'up...","[{'title': 'WormBase 2012: more genomes, more ..."



=== Выбранные столбцы ===


,display_name,num_sequences,num_organisms,url
0,NONCODE,234669,7,http://www.noncode.org/
1,VEGA,0,0,http://vega.sanger.ac.uk/
2,GENCODE,47677,2,http://gencodegenes.org/
3,ENA,12086180,814855,https://www.ebi.ac.uk/ena/browser/
4,TAIR,4406,1,http://www.arabidopsis.org/
5,RefSeq,120355,22524,http://www.ncbi.nlm.nih.gov/refseq/
6,GeneCards,425357,1,https://www.genecards.org/
7,RDP,4779,2487,http://rdp.cme.msu.edu/
8,LNCipedia,126876,1,http://www.lncipedia.org/
9,WormBase,25550,1,http://www.wormbase.org/


Соединение закрыто.
